# Task 2, Step 2: Data Preparation and the Curriculum

Three components, from COCO 2014 and VQAv2:
- **Captions:** 6,000 training images with 3 captions each, 96 development images with 1.
- **Annotation-derived grounding:** programmatic presence and counting pairs.
- **Natural VQAv2:** complementary pairs with different normalized consensus answers and at least 8 of 10 annotators agreeing on each image.

A complementary pair asks one question of two images with different answers. Question wording alone cannot identify both answers. Evaluation images are kept out of training and replay.

In [ ]:
import os, sys, json
root = os.getcwd()
while not os.path.isdir(os.path.join(root, "task2_vlm_grounding")):
    root = os.path.dirname(root)
sys.path.insert(0, root)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display, Markdown

from task2_vlm_grounding.src.notebook_utils import *
setup_notebook()

In [ ]:
from collections import Counter
from task2_vlm_grounding.src.data import check_split_isolation, all_images
from task2_vlm_grounding.src.sampler import CurriculumSampler, STAGE_MIXTURES
manifest = load_manifest()
check_split_isolation(manifest)
print("Split isolation verified. Selected images:", len(all_images(manifest)))
table = pd.DataFrame([
    {"Dataset": "COCO captions", "Training": f"{len(manifest['caption_train'])} examples", "Development": len(manifest["caption_dev"]), "Final test": "-"},
    {"Dataset": "Annotation-derived grounding", "Training": f"{len(manifest['presence_pool'])} presence + {len(manifest['counting_pool'])} counting in pools", "Development": f"{len(manifest['grounding_dev'])} pairs", "Final test": f"{len(manifest['grounding_test'])} pairs"},
    {"Dataset": "Natural VQAv2", "Training": f"{len(manifest['vqa_train'])} pairs", "Development": f"{len(manifest['vqa_dev'])} pairs", "Final test": f"{len(manifest['vqa_test'])} pairs"},
])
table.to_csv(f"{RESULTS_DIR}/dataset_sizes.csv", index=False)
display(table)

## 2.1 What survives the VQAv2 filters?

In [ ]:
stats = pd.DataFrame(manifest["filter_stats"]).T
display(stats)
fig, axes = plt.subplots(1, 3, figsize=(16, 4.2))
stats[["low_agreement", "same_consensus", "retained"]].plot(kind="bar", stacked=True, ax=axes[0], edgecolor="black", rot=0, color=["#e74c3c", "#f39c12", "#2ecc71"])
axes[0].set(title="VQAv2 complementary pair filter", ylabel="Pairs")
types = pd.DataFrame({name: Counter(p["answer_type"] for p in manifest[name]) for name in ["vqa_train", "vqa_dev", "vqa_test"]}).T
types.plot(kind="bar", ax=axes[1], edgecolor="black", rot=0)
axes[1].set(title="Selected natural pairs by answer type", ylabel="Pairs", yscale="log")
category = Counter(p["category"] for p in manifest["grounding_test"])
pd.Series(category).sort_values().tail(20).plot(kind="barh", ax=axes[2], color="#3498db", edgecolor="black")
axes[2].set(title="Grounding final-test pairs per category", xlabel="Pairs")
save_figure("task2_2_data_composition")

## 2.2 Counts in the counting questions
Inspect labels independently of model predictions: COCO annotations can omit objects, and crowd annotations were excluded.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.8))
for ax, name in zip(axes, ["counting_pool", "grounding_test"]):
    answers = Counter(p[s]["target"] for p in manifest[name] if p["type"] == "counting" for s in ("a", "b"))
    pd.Series(answers).sort_index().plot(kind="bar", ax=ax, color="#9b59b6", edgecolor="black", rot=0)
    ax.set(title=name, xlabel="Reference count", ylabel="Question sides")
save_figure("task2_2_counting_answers")

## 2.3 Complementary pairs
**Selection rule:** the first pair of each kind in the final-test lists, shown before any model sees them.

In [ ]:
picks = [next(p for p in manifest["grounding_test"] if p["type"] == "presence"), next(p for p in manifest["grounding_test"] if p["type"] == "counting")] + [next(p for p in manifest["vqa_test"] if p["answer_type"] == t) for t in ["yes/no", "other"]]
fig, axes = plt.subplots(len(picks), 2, figsize=(8, 3.4 * len(picks)))
for row, pair in zip(axes, picks):
    for ax, side in zip(row, ("a", "b")):
        show_image(ax, pair[side]["image"], f"{pair['question'][:60]}\nAnswer: {pair[side]['target']}")
save_figure("task2_2_example_pairs")

## 2.4 The curriculum mixture
The real sampler, drawn with its training seed for the stage's number of draws. There are no exact quotas, so observed shares vary slightly from the targets. Each draw gives two examples.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 3.8))
rows = []
for ax, stage, updates in zip(axes, [2, 3], [1500, 750]):
    sampler = CurriculumSampler(stage, manifest, 6304 + stage)
    draws = updates * 8
    for _ in range(draws):
        sampler.draw()
    frame = pd.DataFrame({"Target": pd.Series(STAGE_MIXTURES[stage]) * 100, "Observed": pd.Series(sampler.counts) / draws * 100})
    frame.plot(kind="bar", ax=ax, edgecolor="black", rot=0)
    ax.set(title=f"Stage {stage}: {draws} draws", ylabel="% of draws")
    rows.append(frame.assign(stage=stage))
pd.concat(rows).to_csv(f"{RESULTS_DIR}/curriculum_sampling.csv")
save_figure("task2_2_curriculum_mixture")